In [ ]:
!nvidia-smi


Mon Feb 16 20:05:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip -q install -U datasets transformers accelerate bitsandbytes sentence-transformers faiss-cpu rank-bm25 tqdm


In [ ]:
import faiss
print("FAISS version:", faiss.__version__)


FAISS version: 1.13.2


In [ ]:
import os, re, math, json, numpy as np
from datasets import load_dataset
from tqdm import tqdm

import faiss
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, CrossEncoder


In [ ]:
MAX_CASES = 6000
CHUNK_CHARS = 1500
CHUNK_OVERLAP = 350

TOPK_DENSE = 60
TOPK_BM25 = 60
TOPK_FUSED = 80
TOPK_RERANK = 6

ALPHA_DENSE = 0.6
ALPHA_BM25 = 0.4


In [ ]:
ds = load_dataset("lex_glue", "scotus", split="train")

cases = []
for row in ds:
    text = row["text"]
    if text and len(text) > 1500:
        cases.append(row)
    if len(cases) >= MAX_CASES:
        break

len(cases)


README.md: 0.00B [00:00, ?B/s]

scotus/train-00000-of-00001.parquet:   0%|          | 0.00/94.4M [00:00<?, ?B/s]

scotus/test-00000-of-00001.parquet:   0%|          | 0.00/40.0M [00:00<?, ?B/s]

scotus/validation-00000-of-00001.parquet:   0%|          | 0.00/39.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1400 [00:00<?, ? examples/s]

4747

In [ ]:
def clean_text(t: str) -> str:
    return re.sub(r"\s+", " ", t).strip()

def chunk_text(t: str, chunk_chars=1500, overlap=250):
    t = clean_text(t)
    chunks = []
    start = 0
    while start < len(t):
        end = min(len(t), start + chunk_chars)
        chunks.append(t[start:end])
        if end == len(t):
            break
        start = max(0, end - overlap)
    return chunks

docs = []
for i, row in enumerate(tqdm(cases)):
    text = row["text"]
    for j, ch in enumerate(chunk_text(text, CHUNK_CHARS, CHUNK_OVERLAP)):
        docs.append({
            "chunk": ch,
            "meta": {"case_id": i, "chunk_id": j}
        })

len(docs)


100%|██████████| 4747/4747 [00:11<00:00, 398.93it/s]


154628

In [ ]:
def tok_bm25(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return [w for w in text.split() if len(w) > 2]

bm25_corpus = [tok_bm25(d["chunk"]) for d in docs]
bm25 = BM25Okapi(bm25_corpus)


In [ ]:
embedder = SentenceTransformer("BAAI/bge-m3", device="cuda")

texts = [d["chunk"] for d in docs]
emb = embedder.encode(
    texts,
    batch_size=128,               # A100 boost
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

index.ntotal


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/1209 [00:00<?, ?it/s]

154628

In [ ]:
def dense_retrieve(query: str, k=60):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    return list(zip(idx[0].tolist(), scores[0].tolist()))

def bm25_retrieve(query: str, k=60):
    scores = bm25.get_scores(tok_bm25(query))
    top = np.argsort(scores)[::-1][:k]
    return list(zip(top.tolist(), scores[top].tolist()))

def fuse(dense, sparse, a=0.60, b=0.40, k=80):
    def norm(items):
        if not items: return {}
        s = np.array([x[1] for x in items], dtype="float32")
        if np.max(s) - np.min(s) < 1e-6:
            s = np.ones_like(s)
        else:
            s = (s - np.min(s)) / (np.max(s) - np.min(s))
        return {items[i][0]: float(s[i]) for i in range(len(items))}

    dmap, smap = norm(dense), norm(sparse)
    all_ids = set(dmap) | set(smap)
    fused = [(i, a*dmap.get(i,0.0) + b*smap.get(i,0.0)) for i in all_ids]
    fused.sort(key=lambda x: x[1], reverse=True)
    return fused[:k]


In [ ]:
reranker = CrossEncoder("BAAI/bge-reranker-large", device="cuda")

def rerank(query: str, fused_results, topk=10):
    pairs = [(query, docs[i]["chunk"]) for i, _ in fused_results]
    scores = reranker.predict(pairs, batch_size=64)  # A100 boost
    scored = [(fused_results[i][0], float(scores[i])) for i in range(len(fused_results))]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:topk]


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="float16")

tok = AutoTokenizer.from_pretrained(GEN_MODEL, use_fast=True)
gen = AutoModelForCausalLM.from_pretrained(GEN_MODEL, quantization_config=bnb, device_map="auto")

SYSTEM = """You are a US Supreme Court legal assistant.
Use ONLY the SOURCES.

Output rules:
- If the SOURCES contain relevant information, answer in 2–5 precise sentences.
- Cite each major claim like [S1], [S2].
- Use at most 3 citations total. Do NOT repeat citations.
- Do NOT output "Not found..." if you used any source content.

If the SOURCES do not contain the answer:
- Output exactly: Not found in the provided corpus.
- Output nothing else.

End with <END>.
"""

def is_smalltalk(q: str) -> bool:
    q = q.strip().lower()
    if len(q) <= 12 and q in {"hi", "hello", "hey", "yo", "sup", "hola"}:
        return True
    if re.fullmatch(r"(hi|hello|hey|good (morning|evening)|how are you)\W*", q):
        return True
    return False

def format_sources(reranked, max_sources=None):
    if max_sources is None:
        max_sources = len(reranked)
    out = []
    for r, (idx, score) in enumerate(reranked[:max_sources], 1):
        m = docs[idx]["meta"]
        head = f"[S{r}] case_id={m['case_id']} chunk={m['chunk_id']}"
        snippet = docs[idx]["chunk"][:900]
        out.append(head + "\n" + snippet)
    return "\n\n".join(out)

def answer(query: str):
    # 0) Smalltalk router
    if is_smalltalk(query):
        return "Hey 👋 Ask me a US Supreme Court law question (e.g., probable cause, Miranda, standing).", []

    # 1) Retrieval
    dense = dense_retrieve(query, TOPK_DENSE)
    sparse = bm25_retrieve(query, TOPK_BM25)
    fused = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr = rerank(query, fused, TOPK_RERANK)

    # 2) Low-score gate (tuned to your score scale: good hits ~0.97)
    if rr and rr[0][1] < 0.75:
        return "Not found in the provided corpus.", rr

    # 3) Build prompt (plain text; avoid apply_chat_template issues)
    sources = format_sources(rr, max_sources=min(TOPK_RERANK, 6))
    prompt = (
        SYSTEM
        + "\n\nQUESTION:\n" + query
        + "\n\nSOURCES:\n" + sources
        + "\n\nANSWER:\n"
    )

    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=4096).to(gen.device)

    with torch.no_grad():
        out = gen.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            use_cache=True
        )

    txt = tok.decode(out[0], skip_special_tokens=True)

    # 4) Keep only the answer region
    if "ANSWER:" in txt:
        txt = txt.split("ANSWER:", 1)[-1]

    # 5) Stop at <END> if present
    if "<END>" in txt:
        txt = txt.split("<END>", 1)[0]

    return txt.strip(), rr


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
import torch

def answer(query: str):
    dense = dense_retrieve(query, TOPK_DENSE)
    sparse = bm25_retrieve(query, TOPK_BM25)
    fused = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr = rerank(query, fused, TOPK_RERANK)

    if rr and rr[0][1] < 0.75:
        return "Not found in the provided corpus.", rr

    sources = format_sources(rr)

    # Build plain-text prompt (no chat template)
    prompt = (
    SYSTEM
    + "\n\nQUESTION:\n" + query
    + "\n\nSOURCES:\n" + sources
    + "\n\nWrite the ANSWER only. End your answer with the token <END>.\n\nANSWER:\n"
    )


    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=3500).to(gen.device)

    with torch.no_grad():
        out = gen.generate(
            **inputs,
            max_new_tokens=450,
            do_sample=False,   # deterministic
            use_cache=True
        )

    text = tok.decode(out[0], skip_special_tokens=True)

    # take only after "ANSWER:"
    if "ANSWER:" in text:
        text = text.split("ANSWER:", 1)[-1]

    # cut at <END>
    if "<END>" in text:
        text = text.split("<END>", 1)[0]

    text = text.strip()
    return text, rr



In [ ]:
resp, rr = answer("probable cause automobile search")
print(resp)
print("Top rerank score:", rr[0][1])


NameError: name 'format_sources' is not defined

In [ ]:
print(len(docs))
len(cases)


154628


4747

In [ ]:
resp, rr = answer("miranda custodial interrogation")
print(resp[:500])
print("Top rerank score:", rr[0][1])


The term "interrogation" under Miranda refers not only to express questioning but also to any words or actions on the part of the police that the police should know are reasonably likely to elicit an incriminating response from the suspect. This definition focuses primarily on the perceptions of the suspect rather than the intent of the police.
Top rerank score: 0.9256383776664734


In [ ]:
def retrieve_ranked_indices(query: str):
    dense = dense_retrieve(query, TOPK_DENSE)
    sparse = bm25_retrieve(query, TOPK_BM25)
    fused_res = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr = rerank(query, fused_res, TOPK_RERANK)
    return [idx for idx, _ in rr]  # ranked chunk indices


In [ ]:
import random

def make_query_from_case(text: str):
    # take first ~250 chars as query (simple, works)
    t = clean_text(text)
    return t[:250]

# Build a mapping case_id -> one chunk index list (for relevance check)
# relevance = any retrieved chunk whose meta case_id == target_case_id


In [ ]:
def recall_at_k(ranked_chunk_ids, target_case_id, k):
    topk = ranked_chunk_ids[:k]
    return 1.0 if any(docs[i]["meta"]["case_id"] == target_case_id for i in topk) else 0.0

def mrr_at_k(ranked_chunk_ids, target_case_id, k):
    topk = ranked_chunk_ids[:k]
    for rank, cid in enumerate(topk, start=1):
        if docs[cid]["meta"]["case_id"] == target_case_id:
            return 1.0 / rank
    return 0.0


In [ ]:
def eval_retrieval(num_samples=200, k_list=(1,3,5,10)):
    idxs = random.sample(range(len(cases)), min(num_samples, len(cases)))
    results = {f"Recall@{k}": [] for k in k_list}
    results.update({f"MRR@{k}": [] for k in k_list})

    for case_id in tqdm(idxs):
        q = make_query_from_case(cases[case_id]["text"])
        ranked = retrieve_ranked_indices(q)

        for k in k_list:
            results[f"Recall@{k}"].append(recall_at_k(ranked, case_id, k))
            results[f"MRR@{k}"].append(mrr_at_k(ranked, case_id, k))

    # averages
    return {m: float(np.mean(v)) for m, v in results.items()}

eval_retrieval(num_samples=200, k_list=(1,3,5,10))


100%|██████████| 200/200 [10:57<00:00,  3.29s/it]


{'Recall@1': 0.995,
 'Recall@3': 1.0,
 'Recall@5': 1.0,
 'Recall@10': 1.0,
 'MRR@1': 0.995,
 'MRR@3': 0.9966666666666666,
 'MRR@5': 0.9966666666666666,
 'MRR@10': 0.9966666666666666}

In [ ]:
queries = [
  "automobile exception probable cause",
  "miranda custodial interrogation",
  "qualified immunity clearly established law",
  "strict scrutiny compelling interest narrowly tailored",
  "commerce clause substantial effects",
  "summary judgment genuine issue of material fact",
  "exclusionary rule good faith exception",
  "standing injury in fact causation redressability",
  "due process procedural due process notice hearing",
  "equal protection rational basis review",
  "first amendment content-based restriction strict scrutiny",
  "time place manner restrictions",
  "search incident to arrest",
  "consent search voluntariness",
  "ineffective assistance of counsel strickland",
  "double jeopardy same offense blockburger",
  "eminent domain public use takings",
  "establishment clause lemon test",
  "free exercise neutral law generally applicable",
  "sixth amendment confrontation clause testimonial",
  "fifth amendment self-incrimination",
  "fourteenth amendment incorporation doctrine",
  "prior restraint first amendment",
  "obscenity miller test",
  "probable cause warrant requirement"
]
len(queries)


25

In [ ]:
def top_sources(query, topn=5):
    dense = dense_retrieve(query, TOPK_DENSE)
    sparse = bm25_retrieve(query, TOPK_BM25)
    fused_res = fuse(dense, sparse, ALPHA_DENSE, ALPHA_BM25, TOPK_FUSED)
    rr = rerank(query, fused_res, topn)
    return rr

def print_sources(query, topn=5, chars=450):
    rr = top_sources(query, topn=topn)
    print("QUERY:", query)
    print("Top rerank score:", rr[0][1] if rr else None)
    print("-"*60)
    for k, (idx, score) in enumerate(rr, 1):
        print(f"[S{k}] score={score:.3f} meta={docs[idx]['meta']}")
        print(docs[idx]["chunk"][:chars])
        print()
    return rr


In [ ]:
max_to_label = 30   # target total

for q in queries:
    if len(labels) >= max_to_label:
        break

    # skip already labeled queries
    if any(l["query"] == q for l in labels):
        continue

    rr = print_sources(q, topn=5, chars=350)
    lab = int(input("Relevant@5? (1 yes / 0 no): "))
    note = input("Notes (optional): ")
    labels.append({"query": q, "relevant_at5": lab, "notes": note})


QUERY: double jeopardy same offense blockburger
Top rerank score: 0.9863648414611816
------------------------------------------------------------
[S1] score=0.986 meta={'case_id': 4377, 'chunk_id': 1}
 respondent for involuntary manslaughter. Pp. 415-421. (a) Whether the offense of failing to reduce speed to avoid an accident is the "same offense" for double jeopardy purposes as the manslaughter charges, depends on whether each statute in question requires proof of a fact which the other does not. Blockburger v. United States, 284 U.S. 299, 52 S

[S2] score=0.979 meta={'case_id': 3993, 'chunk_id': 6}
6 (1975); Gore v. United States, 357 U.S. 386, 78 S.Ct. 1280, 2 L.Ed.2d 1405 (1958). The Blockburger test has its primary relevance in the double jeopardy context, where it is a guide for determining when two separately defined crimes constitute the "same offense" for double jeopardy purposes. Brown v. Ohio, supra.5 7 Cases in which the Government 

[S3] score=0.975 meta={'case_id': 3917, 

In [ ]:
import numpy as np

rel = [x["relevant_at5"] for x in labels]
score = float(np.mean(rel))

print("Concept-Relevance@5:", score)
print("Total labeled queries:", len(rel))


Concept-Relevance@5: 0.92
Total labeled queries: 25


In [ ]:
resp, _ = answer("prior restraint first amendment")
print(resp)


The First Amendment protects against prior restraints on publication, which are considered the essence of censorship and are generally prohibited. The only exception is during wartime when the government may prevent actual obstruction to its recruiting service or similar activities.


In [ ]:
!pip install gradio


In [ ]:
import gradio as gr

def chat_fn(question):
    resp, _ = answer(question)
    return resp

demo = gr.Interface(
    fn=chat_fn,
    inputs=gr.Textbox(lines=2, placeholder="Ask Elite..."),
    outputs="text",
    title="Ai Legal RAG Assistant",
    description="Hybrid Retrieval (BM25 + BGE-M3) + Cross-Encoder Reranking + Grounded Generation",
    examples=[
        "probable cause warrant requirement",
        "prior restraint first amendment",
        "double jeopardy blockburger test",
        "miller test obscenity",
        "qualified immunity clearly established law"
    ]
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c847343df53bf5713b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.launch(share=True)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://12413d1839575702e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive
